## Preparação

Importações e Preparação dos Dados(retirado do `machine_learning.ipynb`)

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    make_scorer,
)


import matplotlib.pyplot as plt

import warnings
import joblib

warnings.filterwarnings('ignore')

df = pd.read_csv('../dados/processado/streaming_retencao_limpo.csv')

df = df.drop(columns=['data_de_assinatura', 'idade_do_usuario', 'horas_consumidas', 'plano', 'pagamento', 'dispositivo_mais_acesso'])

df = pd.get_dummies(df, columns=['genero_favorito'], drop_first=False, dtype=int)

scaler = StandardScaler()
colunas_numericas = ['dias_desde_ultimo_acesso', 'series_concluidas', 'series_abandonadas', 'tickets_suporte_abertos']
df[colunas_numericas] = scaler.fit_transform(df[colunas_numericas])

X = df.drop(columns=["cancelou"])
y = df["cancelou"]

X_train, X_test, y_train, y_test = train_test_split(
    X, 
    y, 
    test_size=0.20, 
    random_state=42, 
    stratify=y
)

# Random Forest
## Grid Search
O modelo é inicializado com `class_weight='balanced'` para balancear e um grid search é feito com foco no scorer recall "Sim"

In [ ]:
modelo_rf = RandomForestClassifier(class_weight='balanced', random_state=42)

param_grid_rf = {
    'n_estimators': [50, 100, 200],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5, 10],
    'criterion': ['gini', 'entropy']
}

scorer_recall = make_scorer(recall_score, pos_label='Sim')

grid_search_rf = GridSearchCV(
    estimator=modelo_rf,
    param_grid=param_grid_rf,
    scoring=scorer_recall, 
    cv=5,                  
    n_jobs=-1,             
    verbose=2              
)

print("Iniciando o Grid Search... Isso testará todas as combinações!")
grid_search_rf.fit(X_train, y_train)

melhor_rf = grid_search_rf.best_estimator_

print(f"Melhores hiperparâmetros encontrados:\n{grid_search_rf.best_params_}")

Melhores hiperparâmetros encontrados:
{'criterion': 'entropy', 'max_depth': 10, 'min_samples_split': 10, 'n_estimators': 100}

# Regressão Logística
## Grid Search
O modelo é inicializado com `class_weight='balanced'` para balancear, com mais interações e um grid search é feito com foco no scorer recall "Sim"

In [ ]:
modelo_lr = LogisticRegression(class_weight='balanced', max_iter=2000, random_state=42)

param_grid_lr = {
    'C': [0.001, 0.01, 0.1, 1, 10, 100],  
    'penalty': ['l1', 'l2'],              
    'solver': ['liblinear', 'saga']       
}

scorer_recall = make_scorer(recall_score, pos_label='Sim')


grid_search_lr = GridSearchCV(
    estimator=modelo_lr,
    param_grid=param_grid_lr,
    scoring=scorer_recall,
    cv=5,
    n_jobs=-1,
    verbose=1 
)

print("Iniciando o Grid Search para Regressão Logística...")
grid_search_lr.fit(X_train, y_train)

melhor_lr = grid_search_lr.best_estimator_

print(f"Melhores hiperparâmetros (Regressão Logística):\n{grid_search_lr.best_params_}")

Melhores hiperparâmetros (Regressão Logística):
{'C': 0.001, 'penalty': 'l2', 'solver': 'liblinear'}

# Exportações dos modelos tunados

In [ ]:
joblib.dump(melhor_rf, '../modelo/melhor_rf_tunada.pkl')
joblib.dump(melhor_lr, '../modelo/melhor_lr_tunada.pkl')

## Análise dos Modelos Tunados

### Random Forest

In [ ]:
y_pred_rf = melhor_rf.predict(X_test)

acuracia = accuracy_score(y_test, y_pred_rf)

print("=" * 40)
print(f"Acurácia do Modelo: {acuracia * 100:.2f}%")
print("=" * 41)

print("\nRelatório Completo (Precision, Recall, F1-Score):")
print(classification_report(y_test, y_pred_rf))

ConfusionMatrixDisplay.from_predictions(y_test, y_pred_rf, cmap='Blues')
plt.title('Matriz de Confusao - Random Forest Tunado')
plt.show()


### Regressão Logística

In [ ]:
y_pred_lr = melhor_lr.predict(X_test)

acuracia = accuracy_score(y_test, y_pred_lr)

print("=" * 40)
print(f"Acurácia do Modelo: {acuracia * 100:.2f}%")
print("=" * 40)

print("\nRelatório Completo (Precision, Recall, F1-Score):")
print(classification_report(y_test, y_pred_lr))

ConfusionMatrixDisplay.from_predictions(y_test, y_pred_lr, cmap='Blues')
plt.title('Matriz de Confusao - Random Forest Tunado')
plt.show()


## Conclusão final

### Modelos Tunados

| Métrica | Regressão logística | Random Forest |
| :---: | :---: | :---: |
| Falsos negativos | 0 | 2 |        
| Falsos positivos | 132 | 33 |
| Recall | 1.00 | 0.88 |
| Precision | 0.11 | 0.30 |


Analizando as métricas dos dois modelos tunados, podemos chegar na conclusão de que ambos obtiveram resultado mais favorável comparado com os modelos antes da tunagem de hiperparâmetros.

Regressão logística: Obteve 100% de eficiência no recall, porém como efeito colateral, preveu errado uma quantidade elevada de positivos, sendo positivos reais no dataset um total de 16 e o modelo preveu 148 positivos.

Random Forest: Obteve um melhor resultado geral, já que o trade-off foi mais suave e, apesar do recall da Regressão Logística ser 'perfeita', possui um recall de 88% e preveu 47 positivos, 100 a menos que o modelo anterior.

Pode-se, portanto, concluir com as métricas do modelo de RF que:

   - Verdadeiro negativo: cliente satisfeito que irá continuar consumindo,
   - Verdadeiro positivo: cliente insatisfeito que com certeza irá cancelar,
   - Falso positivo: cliente com tendência ao cancelamento, mas que continua usando o serviço,
   - Falso negativo: cliente com comportamento de insatisfação, mas que está satisfeito.